# Training District-level Models
This notebook contains the code used to train the machine learning models on the new 2001-2012 district-level dataset.

## Part 1: Regression Model
Predicting the exact count of crimes using a `RandomForestRegressor`.


In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVR
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

# Setup paths
project_root = Path(os.getcwd()).resolve()
data_path = project_root / "data" / "processed" / "crime_data_district_long.csv"
model_path = project_root / "models"
model_path.mkdir(exist_ok=True)

print(f"Data path: {data_path}")

# Load data
df = pd.read_csv(data_path)

# Exclude total rows
df = df[df['Crime Description'] != 'TOTAL IPC CRIMES']

# Encode categorical features for the pipeline
state_encoder = LabelEncoder()
district_encoder = LabelEncoder()
crime_encoder = LabelEncoder()

df['State_Enc'] = state_encoder.fit_transform(df['State'])
df['District_Enc'] = district_encoder.fit_transform(df['District'])
df['Crime_Enc'] = crime_encoder.fit_transform(df['Crime Description'])

# Feature Engineering (Yearly Lags)
df = df.sort_values(by=['State', 'District', 'Crime Description', 'Year'])
grouper = df.groupby(['State', 'District', 'Crime Description'])['Crime Count']

df['Lag_1'] = grouper.shift(1)
df['Lag_2'] = grouper.shift(2)
df['Rolling_Mean_2'] = grouper.transform(lambda x: x.shift(1).rolling(2).mean())

df = df.dropna()
print(f"Cleaned training data: {len(df)} rows")

# Split
train_df = df[df['Year'] < 2012]
test_df = df[df['Year'] == 2012]

X_train = train_df[['State_Enc', 'District_Enc', 'Crime_Enc', 'Year', 'Lag_1', 'Lag_2', 'Rolling_Mean_2']]
y_train = train_df['Crime Count']

# Build pipeline (No OneHot for 808 districts to save memory)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['Year', 'Lag_1', 'Lag_2', 'Rolling_Mean_2', 'State_Enc', 'District_Enc', 'Crime_Enc'])
    ]
)

# ... (inside pipeline) ...
reg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=50, max_depth=15, random_state=42, n_jobs=-1))
])

log_transformer = FunctionTransformer(
    func=np.log1p,
    inverse_func=np.expm1,
    validate=False
)

model = TransformedTargetRegressor(
    regressor=reg_pipeline,
    transformer=log_transformer
)

# Train
print(f"Training RandomForest on {len(X_train)} rows...")
model.fit(X_train, y_train)
print("Training Complete.")

# Evaluate
print("Evaluating model on 2012 Test Set...")
y_test = test_df['Crime Count']
X_test = test_df[['State_Enc', 'District_Enc', 'Crime_Enc', 'Year', 'Lag_1', 'Lag_2', 'Rolling_Mean_2']]

y_pred = model.predict(X_test)
y_pred = np.maximum(0, np.round(y_pred)) # Ensure counts are non-negative integers

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n=== Model Evaluation (Regression Metrics) ===")
print(f"R-squared (R2 Score): {r2:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")

# Calculate a pseudo-accuracy for the user (percentage of predictions within a small margin of error)
# For counts, we can classify a prediction as "accurate" if it's within +/- 15% of the actual value or +/- 5 counts (for small numbers)
margin = np.maximum(5, 0.15 * y_test)
accurate_preds = np.abs(y_test - y_pred) <= margin
pseudo_accuracy = np.mean(accurate_preds) * 100
print(f"Pseudo-Accuracy (within 15% or 5 crimes margin): {pseudo_accuracy:.2f}%")
print("=============================================\n")

# Save
joblib.dump(model, model_path / "model.pkl")
joblib.dump(state_encoder, model_path / "state_encoder.pkl")
joblib.dump(district_encoder, model_path / "district_encoder.pkl")
joblib.dump(crime_encoder, model_path / "crime_encoder.pkl")

print(f"All files saved successfully in '{model_path}' folder!")



## Part 2: Classification Model (Crime Trend)
Predicting whether a specific crime will **increase** in the following year using a `RandomForestClassifier`. This provides Accuracy, Precision, Recall, F1, and AUC metrics.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve
import joblib

# Setup paths
project_root = Path(os.getcwd()).resolve()
data_path = project_root / "data" / "processed" / "crime_data_district_long.csv"
model_path = project_root / "models"
model_path.mkdir(exist_ok=True)

print(f"Data path: {data_path}")

# Load data
df = pd.read_csv(data_path)

# Exclude total rows
df = df[df['Crime Description'] != 'TOTAL IPC CRIMES']

# Encode categorical features
state_encoder = LabelEncoder()
district_encoder = LabelEncoder()
crime_encoder = LabelEncoder()

df['State_Enc'] = state_encoder.fit_transform(df['State'])
df['District_Enc'] = district_encoder.fit_transform(df['District'])
df['Crime_Enc'] = crime_encoder.fit_transform(df['Crime Description'])

# Feature Engineering (Yearly Lags)
df = df.sort_values(by=['State', 'District', 'Crime Description', 'Year'])
grouper = df.groupby(['State', 'District', 'Crime Description'])['Crime Count']

df['Lag_1'] = grouper.shift(1)
df['Lag_2'] = grouper.shift(2)
df['Lag_3'] = grouper.shift(3)
df['Lag_4'] = grouper.shift(4)

df['Rolling_Mean_2'] = grouper.transform(lambda x: x.shift(1).rolling(2).mean())
df['Rolling_Mean_3'] = grouper.transform(lambda x: x.shift(1).rolling(3).mean())
df['Rolling_Mean_4'] = grouper.transform(lambda x: x.shift(1).rolling(4).mean())

df = df.dropna()

# --- Convert to Classification Task ---
# Target: Will crime count be strictly greater than last year? (1 = Increase, 0 = Decrease/Same)
df['Crime_Trend'] = (df['Crime Count'] > df['Lag_1']).astype(int)

print(f"Cleaned training data: {len(df)} rows")
print("Target Class Distribution:")
print(df['Crime_Trend'].value_counts(normalize=True))

# Split
train_df = df[df['Year'] < 2012]
test_df = df[df['Year'] == 2012]

X_train = train_df[['State_Enc', 'District_Enc', 'Crime_Enc', 'Year', 'Lag_1', 'Lag_2', 'Lag_3', 'Lag_4', 'Rolling_Mean_2', 'Rolling_Mean_3', 'Rolling_Mean_4']]
y_train = train_df['Crime_Trend']

X_test = test_df[['State_Enc', 'District_Enc', 'Crime_Enc', 'Year', 'Lag_1', 'Lag_2', 'Lag_3', 'Lag_4', 'Rolling_Mean_2', 'Rolling_Mean_3', 'Rolling_Mean_4']]
y_test = test_df['Crime_Trend']

# Build pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['Year', 'Lag_1', 'Lag_2', 'Lag_3', 'Lag_4', 'Rolling_Mean_2', 'Rolling_Mean_3', 'Rolling_Mean_4', 'State_Enc', 'District_Enc', 'Crime_Enc'])
    ]
)

# Using balanced class weight to handle slight imbalances, and optimizing depth for better accuracy/AUC
clf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=200, max_depth=25, min_samples_split=5, random_state=42, n_jobs=-1, class_weight='balanced'))
])

# Train
print(f"Training RandomForestClassifier on {len(X_train)} rows...")
clf_pipeline.fit(X_train, y_train)
print("Training Complete.")

# Evaluate
print("\n=== Model Evaluation (Classification Metrics) ===")
y_pred = clf_pipeline.predict(X_test)
y_prob = clf_pipeline.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"ROC AUC:   {auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Decrease/Stable (0)', 'Increase (1)']))

# Save ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) - Crime Trend Prediction')
plt.legend(loc="lower right")

output_img = project_root / "roc_curve.png"
plt.savefig(output_img)
print(f"\nROC curve saved to {output_img}")

# Save Classifier
joblib.dump(clf_pipeline, model_path / "trend_classifier.pkl")
print(f"Classifier saved successfully to {model_path / 'trend_classifier.pkl'}")

